# Trabajo Práctico Integrador – Entrega 1

**Materia:** Introducción al Análisis de Datos  
**Tema:** Cancelación de reservas hoteleras  
**Comisión:** 10 (Grupo J)  
**Integrante:** Nicolás Viruel  
**Entrega:** 1 – Unidad N° 1  
**Año:** 2026


## 1. Presentación del problema

El hotel necesita entender **por qué se cancelan las reservas** antes del check-in. Cada registro del dataset representa una reserva con características como tipo de hotel, anticipación (`lead_time`), canal de distribución, tipo de depósito y la variable objetivo `is_canceled`.

Desde el ciclo de vida del análisis, estamos en una **fase inicial de comprensión y exploración**: todavía no modelamos ni predecimos, pero sí convertimos datos crudos en **información** (estructura, conteos, promedios) que después puede convertirse en **conocimiento** si se interpreta en contexto.

**Variable objetivo:** `is_canceled` (0 = no cancelada, 1 = cancelada).

**Preguntas que orientan este primer avance:**
1. ¿Qué porcentaje de reservas se cancela?
2. ¿Qué variables numéricas y categóricas tenemos disponibles?
3. ¿Cómo se distribuyen los tipos de hotel y los canales de reserva?
4. ¿Qué rangos presentan variables como `lead_time` o `adr`?


## 2. Dataset asignado

- **Comisión:** 10  
- **Archivo:** `hotel_booking_TPI_grupo_J.csv`  
- **Origen:** subset asignado por la cátedra (Hotel Booking Demand)

> Subí el CSV en la misma carpeta que este notebook (Colab/Drive) antes de ejecutar la carga.


In [1]:
import pandas as pd

archivo = "hotel_booking_TPI_grupo_J.csv"
df = pd.read_csv(archivo)
print("Dataset cargado:", archivo)


Dataset cargado: hotel_booking_TPI_grupo_J.csv


## 3. Vista inicial del dataset

In [2]:
df.head()

,booking_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,arrival_date,stays_in_weekend_nights,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,HB-035978,Resort Hotel,0,261,2025,April,18,30,2025-04-30,2,...,A,0,No Deposit,40.0,NaN,0,Contract,42.95,0,1
1,HB-004892,Resort Hotel,1,85,2024,April,14,7,2024-04-07,0,...,A,0,No Deposit,67.0,NaN,0,Transient-Party,64.00,0,0
2,HB-076102,City Hotel,1,364,2023,October,42,16,2023-10-16,0,...,A,0,Non Refund,6.0,NaN,0,Transient-Party,101.50,0,0
3,HB-102752,City Hotel,0,32,2024,December,49,4,2024-12-04,2,...,D,0,No Deposit,9.0,NaN,0,Transient,114.00,0,0
4,HB-069831,City Hotel,1,126,2025,June,23,7,2025-06-07,0,...,A,0,No Deposit,27.0,NaN,0,Transient,89.10,0,1


## 4. Estructura general

In [3]:
filas, columnas = df.shape
print(f"Filas (reservas): {filas}")
print(f"Columnas (variables): {columnas}")
print("\nNombres de columnas:")
print(df.columns.tolist())


Filas (reservas): 25000
Columnas (variables): 32

Nombres de columnas:
['booking_id', 'hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'arrival_date', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests']


## 5. Tipos de datos

In [4]:
print(df.dtypes)
print("\n--- info() ---")
df.info()


booking_id                            str
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
arrival_date                          str
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                   

## 6. Clasificación inicial de variables

Clasificación manual según el dominio del problema:


In [5]:
numericas = [
    "is_canceled", "lead_time", "arrival_date_year", "arrival_date_week_number",
    "arrival_date_day_of_month", "stays_in_weekend_nights", "stays_in_week_nights",
    "adults", "children", "babies", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes", "days_in_waiting_list",
    "adr", "required_car_parking_spaces", "total_of_special_requests",
]
categoricas = [
    "booking_id", "hotel", "arrival_date_month", "arrival_date", "meal", "country",
    "market_segment", "distribution_channel", "reserved_room_type", "assigned_room_type",
    "deposit_type", "customer_type",
]
temporales = ["arrival_date_year", "arrival_date_month", "arrival_date", "arrival_date_week_number", "arrival_date_day_of_month"]

print("Numéricas:", len(numericas))
print("Categóricas:", len(categoricas))
print("Temporales (también presentes en el dataset):", temporales)


Numéricas: 17
Categóricas: 12
Temporales (también presentes en el dataset): ['arrival_date_year', 'arrival_date_month', 'arrival_date', 'arrival_date_week_number', 'arrival_date_day_of_month']


## 7. Caracterización descriptiva inicial

In [6]:
df.describe().round(2)


,is_canceled,lead_time,arrival_date_year,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests
count,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.0,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,25000.00,21521.00,1440.00,25000.00,25000.00,25000.00,25000.00
mean,0.37,104.22,2024.16,26.69,15.81,0.92,2.5,1.85,0.10,0.01,0.03,0.09,0.14,0.23,86.43,187.33,2.28,101.68,0.06,0.57
std,0.48,107.10,0.71,13.41,8.75,1.00,1.9,0.57,0.39,0.09,0.18,0.87,1.61,0.64,111.05,128.59,17.08,47.97,0.24,0.79
min,0.00,0.00,2023.00,1.00,1.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,9.00,0.00,0.00,0.00,0.00
25%,0.00,18.00,2024.00,16.00,8.00,0.00,1.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,9.00,62.00,0.00,69.03,0.00,0.00
50%,0.00,69.00,2024.00,27.00,16.00,1.00,2.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,14.00,186.00,0.00,94.50,0.00,0.00
75%,1.00,160.00,2025.00,37.00,23.00,2.00,3.0,2.00,0.00,0.00,0.00,0.00,0.00,0.00,229.00,268.00,0.00,126.00,0.00,1.00
max,1.00,629.00,2025.00,52.00,31.00,18.00,42.0,40.00,3.00,2.00,1.00,26.00,72.00,18.00,535.00,541.00,391.00,387.00,3.00,5.00


In [7]:
print("Distribución de la variable objetivo:")
print(df["is_canceled"].value_counts())
print(f"\nPorcentaje de cancelaciones: {df['is_canceled'].mean() * 100:.2f}%")


Distribución de la variable objetivo:
is_canceled
0    15741
1     9259
Name: count, dtype: int64

Porcentaje de cancelaciones: 37.04%


In [8]:
print("Tipo de hotel:")
print(df["hotel"].value_counts())

print("\nCanal de distribución:")
print(df["distribution_channel"].value_counts().head())


Tipo de hotel:
hotel
City Hotel      16614
Resort Hotel     8386
Name: count, dtype: int64

Canal de distribución:
distribution_channel
TA/TO        20384
Direct        3140
Corporate     1431
GDS             45
Name: count, dtype: int64


## 8. Primeras observaciones

- El dataset tiene **25.000 reservas** y **32 variables**, lo que permite un análisis exploratorio inicial amplio.
- Aproximadamente **37%** de las reservas están canceladas (`is_canceled = 1`), un nivel relevante para el negocio.
- Hay dos tipos de hotel (**City Hotel** y **Resort Hotel**); conviene comparar cancelaciones entre ambos en entregas futuras.
- `lead_time` varía desde reservas de último momento hasta más de 600 días de anticipación; la media ronda los 100 días (según `describe()`).
- Variables como `agent` y `company` tienen muchos valores faltantes (se observa en `info()`), lo que limitará su uso directo sin tratamiento posterior.


## 9. Consideraciones éticas iniciales

- Los datos corresponden a reservas reales anonimizadas; no deben usarse para identificar huéspedes ni compartirse fuera del ámbito académico.
- En esta etapa **no afirmo causas** (por ejemplo, que un canal "provoca" cancelaciones): solo describo patrones que después deberán validarse con más análisis.
- Cualquier recomendación al hotel debería basarse en evidencia acumulada y considerar el impacto en clientes (políticas de depósito, sobreventa, etc.).
